# FCBillar — Anàlisi bàsica

Notebook d'exemple per consultar la BD SQLite local. Assumeix que has fet ja:

```powershell
uv run fcbillar import-clubs
uv run fcbillar sync
uv run fcbillar backfill 0 --historical
```

Si la BD està buida, totes les taules retornaran zero files.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

# La BD s'ubica a `data/fcbillar.db` per defecte (vegeu config.py).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "fcbillar.db"
assert DB_PATH.exists(), f"BD no existeix a {DB_PATH}; executa `uv run fcbillar init-db` primer."
conn = sqlite3.connect(DB_PATH)
print(f"Connectat a {DB_PATH}")

## Estadístiques generals

In [ ]:
tables = [
    "clubs", "players", "modalitats", "competicions", "rankings",
    "ranking_entries", "games", "ranking_game_links",
    "temporades", "equips", "encontres_lliga", "club_aliases",
]
counts = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tables}
pd.Series(counts, name="files").to_frame()

## Top jugadors per modalitat (rànquing més recent de cada modalitat)

In [ ]:
top_per_modalitat = pd.read_sql_query(
    """
    SELECT m.nom AS modalitat, e.posicio, p.nom AS jugador, p.fcb_id,
           e.mitjana_general AS mitjana_jugador,
           json_extract(e.extras_json, '$.mitjana_contraris') AS mitjana_contraris,
           json_extract(e.extras_json, '$.definitiva') AS def
    FROM ranking_entries e
    JOIN rankings r ON r.id = e.ranking_id
    JOIN modalitats m ON m.id = r.modalitat_id
    JOIN players p ON p.id = e.player_id
    WHERE r.num_seq = (SELECT MAX(r2.num_seq) FROM rankings r2 WHERE r2.modalitat_id = r.modalitat_id)
      AND e.posicio <= 5
    ORDER BY m.codi_fcb, e.posicio
    """,
    conn,
)
top_per_modalitat

## Distribució de partides per categoria de competició

In [ ]:
pd.read_sql_query(
    """
    SELECT c.nom AS competicio, COUNT(*) AS partides
    FROM games g JOIN competicions c ON c.id = g.competicio_id
    GROUP BY c.nom ORDER BY partides DESC
    """,
    conn,
)

## Partides per club (només les que tenen club assignat via lliga)

In [ ]:
pd.read_sql_query(
    """
    WITH club_games AS (
        SELECT c1.nom AS club, g.id AS game_id
        FROM games g JOIN equips e1 ON e1.id = g.equip1_id JOIN clubs c1 ON c1.id = e1.club_id
        WHERE g.equip1_id IS NOT NULL
        UNION ALL
        SELECT c2.nom AS club, g.id AS game_id
        FROM games g JOIN equips e2 ON e2.id = g.equip2_id JOIN clubs c2 ON c2.id = e2.club_id
        WHERE g.equip2_id IS NOT NULL
    )
    SELECT club, COUNT(DISTINCT game_id) AS partides
    FROM club_games
    GROUP BY club
    ORDER BY partides DESC
    LIMIT 15
    """,
    conn,
)

## Jugadors seguits + comptador de partides

In [ ]:
pd.read_sql_query(
    """
    SELECT p.nom, p.fcb_id,
           (SELECT COUNT(*) FROM games g WHERE g.player1_id = p.id OR g.player2_id = p.id) AS partides
    FROM players p
    WHERE p.seguiment = 1
    ORDER BY partides DESC
    """,
    conn,
)